# 4D SU(2) validated RG — M1–M6 plan and tracker

This notebook tracks implementation and mathematical certification milestones.
It does not perform the RG calculation itself.

Use it from persistent storage. The tracker state is stored as JSON so it survives
GPU runtime shutdown.

## Governing documents

- `validated_4d_su2_rg_codex_design_v0_2.md`
- `M1_M6_VALIDATED_RG_ROADMAP.md`
- `MATHEMATICAL_CERTIFICATION_SPEC.md`
- `CODEX_PROMPTS_M1_M6.md`
- `AGENTS_validated_4d_su2_rg_v0_2.md`

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

Status = Literal[
    "NOT_STARTED", "IN_PROGRESS", "BLOCKED_MATH", "BLOCKED_RESOURCE",
    "TEST_FAILED", "ACCEPTED"
]

@dataclass
class Milestone:
    name: str
    status: Status = "NOT_STARTED"
    sessions_completed: int = 0
    acceptance_checks: dict[str, bool] = field(default_factory=dict)
    rigorous_outputs: list[str] = field(default_factory=list)
    heuristic_outputs: list[str] = field(default_factory=list)
    blockers: list[str] = field(default_factory=list)
    next_actions: list[str] = field(default_factory=list)
    updated_at: str = ""

@dataclass
class PlanState:
    schema_version: int = 1
    project: str = "validated_4d_su2_rg"
    current_milestone: str = "M1"
    certification_status: str = "NOT_CERTIFIED"
    accepted_parent_m0: dict[str, str] = field(default_factory=lambda: {
        "parent_milestone": "M0",
        "parent_run_id": "20260719T120406Z-731966c8fd28",
        "parent_checkpoint": "ckpt_000014",
        "parent_checkpoint_path": "/storage/validated_4d_su2_rg/runs/20260719T120406Z-731966c8fd28/checkpoints/ckpt_000014",
        "acceptance_scope": "user-provided report; no independent checkpoint audit",
    })
    milestones: dict[str, Milestone] = field(default_factory=dict)
    updated_at: str = ""


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def default_state() -> PlanState:
    names = {
        "M1": ["2D exact recurrence", "analytic value tail", "analytic gradient tail", "resume regression"],
        "M2": ["dense reference", "armillary generator", "basis equivalence", "symmetry tests"],
        "M3": ["matrix-free matvec", "adjoint test", "RSVD pilot", "GPU resume", "OOM recovery"],
        "M4": ["forward tangent", "basis residual", "finite-difference regression", "error ledger"],
        "M5": ["P1-P11 gates", "normalization lower bound", "interval influence", "independent verifier"],
        "M6": ["multi-step balls", "P12-P13 gates", "final weighted matrix", "final verifier", "q_cert < 1"],
    }
    milestones = {
        key: Milestone(name=key, acceptance_checks={item: False for item in checks})
        for key, checks in names.items()
    }
    return PlanState(milestones=milestones, updated_at=utc_now())


## Persistent tracker path

Set `VALIDATED_RG_PERSIST_ROOT` to the same persistent root used by the main driver.

In [ ]:
def tracker_path() -> Path:
    raw = os.environ.get("VALIDATED_RG_PERSIST_ROOT")
    if not raw:
        raise RuntimeError("Set VALIDATED_RG_PERSIST_ROOT to persistent storage.")
    path = Path(raw).expanduser().resolve() / "project" / "M1_M6_plan_state.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def save_state(state: PlanState) -> Path:
    state.updated_at = utc_now()
    path = tracker_path()
    tmp = path.with_suffix(path.suffix + ".tmp")
    payload = asdict(state)
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)
    return path


def load_state() -> PlanState:
    path = tracker_path()
    if not path.exists():
        state = default_state()
        save_state(state)
        return state
    raw = json.loads(path.read_text(encoding="utf-8"))
    raw["milestones"] = {
        key: Milestone(**value) for key, value in raw["milestones"].items()
    }
    return PlanState(**raw)

# state = load_state()
# state


In [ ]:
def mark_check(state: PlanState, milestone: str, check: str, value: bool = True) -> None:
    ms = state.milestones[milestone]
    if check not in ms.acceptance_checks:
        raise KeyError(f"Unknown acceptance check: {check}")
    ms.acceptance_checks[check] = value
    ms.updated_at = utc_now()
    ms.status = "ACCEPTED" if all(ms.acceptance_checks.values()) else "IN_PROGRESS"
    save_state(state)


def add_session_summary(
    state: PlanState,
    milestone: str,
    *,
    rigorous: list[str] | None = None,
    heuristic: list[str] | None = None,
    blockers: list[str] | None = None,
    next_actions: list[str] | None = None,
) -> None:
    ms = state.milestones[milestone]
    ms.sessions_completed += 1
    ms.rigorous_outputs.extend(rigorous or [])
    ms.heuristic_outputs.extend(heuristic or [])
    ms.blockers = blockers or []
    ms.next_actions = next_actions or []
    ms.updated_at = utc_now()
    if ms.status == "NOT_STARTED":
        ms.status = "IN_PROGRESS"
    save_state(state)


## Certification gate

The tracker must never set `CERTIFIED` automatically from approximate metrics.
Set it only after the independent M6 verifier confirms all P0–P13 gates and
`q_cert_upper < 1`.

In [ ]:
def set_final_certification(
    state: PlanState,
    *,
    all_proof_gates_passed: bool,
    independent_verifier_passed: bool,
    q_cert_upper: str,
) -> None:
    from decimal import Decimal

    q = Decimal(q_cert_upper)
    if not all_proof_gates_passed:
        raise ValueError("All mathematical proof gates must pass.")
    if not independent_verifier_passed:
        raise ValueError("Independent verifier must pass.")
    if not q < Decimal(1):
        raise ValueError("q_cert_upper must be strictly below one.")
    if state.milestones["M6"].status != "ACCEPTED":
        raise ValueError("M6 acceptance checks are incomplete.")
    state.certification_status = "CERTIFIED"
    save_state(state)


## Per-session usage

At the beginning of each GPU session:

```python
state = load_state()
state.current_milestone
```

At the end, record rigorous outputs, heuristic outputs, blockers, and the exact next
work items before the main runner performs its final checkpoint.